# Notebook de definición del problema y entorno reproducible — Fase 1

**Proyecto:** Mantenimiento predictivo con AI4I 2020  
**Asignatura:** Programación para la Ciencia de Datos  
**Grupo:** 9  
**Integrante:** Alexander Sepulveda Ormazabal  

## ¿Qué produce la Fase 1?

La Fase 1 establece las bases conceptuales y técnicas del proyecto. En ella se
define el problema que será estudiado, se documenta el conjunto de datos, se
comprueba el entorno de ejecución y se organiza el repositorio que sostendrá
las fases posteriores.

El flujo de este cuaderno es:

**Definir → Configurar → Verificar → Estructurar → Documentar → Versionar**

La Fase 1 reconoce y valida inicialmente los datos, pero no realiza su limpieza,
imputación, codificación ni escalamiento. Esas tareas corresponden a la Fase 2.



## Introducción

Las fallas inesperadas en maquinaria industrial pueden generar interrupciones
operacionales, pérdidas de productividad y mayores necesidades de mantenimiento.
El conjunto de datos AI4I 2020 contiene información sobre condiciones de operación
de maquinaria, como temperatura, velocidad de rotación, torque y desgaste de
herramienta, además del registro de ocurrencia de fallas.

El proyecto busca estudiar la relación entre estas condiciones operacionales y
las fallas, identificar posibles patrones de riesgo y evaluar posteriormente si
las variables disponibles permiten anticipar su ocurrencia.
    

## Problemática

Las fallas inesperadas en maquinaria industrial pueden ocasionar detenciones no
planificadas, pérdidas de productividad y mayores necesidades de mantenimiento.
Aunque existen mediciones de las condiciones de operación de los equipos, es
necesario determinar cuáles de ellas se relacionan con la ocurrencia de fallas.

El proyecto utilizará el conjunto de datos AI4I 2020 para estudiar variables como
la temperatura del aire y del proceso, la velocidad de rotación, el torque, el
desgaste de herramienta y el tipo de producto. A partir de estos datos se buscará
identificar condiciones operacionales asociadas con fallas y posibles patrones
de riesgo, considerando que el conjunto es sintético y que las asociaciones
observadas no demuestran por sí solas relaciones causales.

## Pregunta de investigación

> ¿Qué condiciones operacionales están asociadas con la ocurrencia de fallas en
> maquinaria industrial y en qué medida estas variables permiten identificar
> situaciones de riesgo de falla?

> ## Objetivo general

Analizar la relación entre las condiciones operacionales de la maquinaria
industrial y la ocurrencia de fallas mediante el conjunto de datos AI4I 2020,
con el propósito de identificar posibles patrones de riesgo y evaluar
posteriormente la capacidad de estas variables para anticipar fallas.


## Objetivos específicos

1. Caracterizar las variables operacionales y la variable de falla presentes
   en el conjunto de datos AI4I 2020.

2. Evaluar la calidad inicial del conjunto mediante la revisión de sus
   dimensiones, tipos de datos, valores faltantes, duplicados, rangos y
   distribución de la variable objetivo.

3. Preparar un conjunto de datos consistente y reproducible, justificando las
   decisiones de limpieza, selección y transformación aplicadas en la Fase 2.

4. Explorar la relación entre la temperatura, la velocidad de rotación, el
   torque, el desgaste de herramienta, el tipo de producto y la ocurrencia
   de fallas.

5. Evaluar en las fases posteriores si las variables operacionales permiten
   reconocer situaciones con mayor riesgo de falla.

## Herramientas del proyecto

En esta sección se importan las librerías y los módulos necesarios para verificar
el entorno, documentar el conjunto de datos y consultar el estado del repositorio.

También se establece una semilla fija para que cualquier procedimiento que
incorpore aleatoriedad pueda repetirse bajo las mismas condiciones.

In [ ]:
import sys
import json
import platform
import subprocess
import shutil
import importlib
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

# Semilla compartida por las distintas fases del proyecto.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 · Grupo 9 ·", date.today().isoformat())
print("Semilla declarada:", SEMILLA)

## 1. Definición del problema

Esta sección registra la problemática, las preguntas, los objetivos, el alcance
y las limitaciones del proyecto en una estructura de datos.

La información coincide con la definición presentada anteriormente, pero se
organiza como un diccionario de Python para poder reutilizarla posteriormente
en las validaciones y en los metadatos del proyecto.

In [ ]:
PROYECTO = {
    "titulo": "Mantenimiento predictivo con AI4I 2020",
    "grupo": "Grupo 9",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Alexander Sepulveda Ormazabal"],

    "problematica": (
        "Las fallas inesperadas en maquinaria industrial pueden provocar "
        "interrupciones operacionales, pérdidas de productividad y mayores "
        "necesidades de mantenimiento. El proyecto busca estudiar qué "
        "condiciones operacionales están asociadas con la ocurrencia de fallas."
    ),

    "objetivo_general": (
        "Analizar la relación entre las condiciones operacionales de la "
        "maquinaria industrial y la ocurrencia de fallas mediante el conjunto "
        "de datos AI4I 2020, con el propósito de identificar posibles patrones "
        "de riesgo y evaluar posteriormente la capacidad de estas variables "
        "para anticipar fallas."
    ),

    "objetivos_especificos": [
        "Caracterizar las variables operacionales y la variable de falla.",
        "Evaluar la calidad inicial del conjunto de datos.",
        "Preparar un conjunto de datos consistente y reproducible en la Fase 2.",
        "Explorar la relación entre las condiciones operacionales y las fallas.",
        "Evaluar posteriormente la capacidad de identificar situaciones de riesgo.",
    ],

    "preguntas": [
        "¿Qué consecuencias generan las fallas inesperadas en la maquinaria?",
        "¿Qué condiciones operacionales registra AI4I 2020?",
        "¿Qué variables podrían estar asociadas con un mayor riesgo de falla?",
        "¿Qué limitaciones presenta el conjunto de datos?",
    ],

    "alcance": {
        "incluye": [
            "Definición del problema",
            "Verificación del entorno reproducible",
            "Descripción y validación inicial del conjunto de datos",
            "Preparación de los datos en la Fase 2",
            "Análisis posterior de las condiciones asociadas con fallas",
        ],
        "excluye": [
            "Interpretación causal de las asociaciones",
            "Validación directa en maquinaria industrial real",
            "Despliegue de un sistema de mantenimiento en producción",
        ],
    },

    "limitaciones": [
        "AI4I 2020 es un conjunto de datos sintético.",
        "Las asociaciones observadas no demuestran causalidad.",
        "La variable objetivo presenta menos fallas que casos sin falla.",
        "Los resultados obtenidos no garantizan desempeño en equipos reales.",
    ],
}

### Presentación y verificación de la definición

La función siguiente comprueba que la configuración contiene el título, la
problemática, el objetivo general y los objetivos específicos. Después presenta
la información principal del proyecto de forma ordenada.

La verificación permite detectar inmediatamente si falta algún elemento requerido.

In [ ]:
def presentar_proyecto(config):
    """
    Comprueba y presenta la definición principal del proyecto.

    Parámetros
    ----------
    config : dict
        Diccionario con la configuración del proyecto.

    Retorna
    -------
    None
    """
    obligatorias = (
        "titulo",
        "problematica",
        "objetivo_general",
        "objetivos_especificos",
    )

    faltantes = [clave for clave in obligatorias if not config.get(clave)]

    if faltantes:
        raise KeyError(
            f"Faltan elementos obligatorios en la configuración: {faltantes}"
        )

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")

    print("\nProblemática")
    print(config["problematica"])

    print("\nObjetivo general")
    print(config["objetivo_general"])

    print("\nObjetivos específicos")
    for numero, objetivo in enumerate(
        config["objetivos_especificos"], start=1
    ):
        print(f"  {numero}. {objetivo}")

    print("\nPreguntas que orientan el proyecto")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")

In [ ]:
presentar_proyecto(PROYECTO)

### Alcance y limitaciones

El alcance establece qué actividades forman parte del proyecto y cuáles quedan
fuera. Las limitaciones indican las condiciones que deben considerarse al
interpretar los resultados.

Declarar estos elementos evita atribuir al análisis conclusiones que los datos
o el método no pueden respaldar

In [ ]:
print("Alcance de la Fase 1")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nLimitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")


## 2. Verificación del entorno reproducible

Antes de utilizar el conjunto de datos se comprueba que el notebook se ejecuta
con el intérprete del entorno virtual del proyecto y que las librerías
científicas necesarias están disponibles.

También se muestra la carpeta de trabajo porque las rutas relativas se resuelven
desde esa ubicación. El intérprete debe encontrarse dentro de `.venv` y la
carpeta de trabajo debe corresponder a la raíz del repositorio.

In [ ]:
import sys
from pathlib import Path
print("Intérprete:", sys.executable)
print("Carpeta de trabajo:", Path.cwd())
for lib in ["numpy", "pandas", "matplotlib", "sklearn"]:
    try:
        print(f" {lib:12}", __import__(lib).__version__)
    except ImportError:
        print(f" {lib:12} NO DISPONIBLE en este kernel")
        
    



In [ ]:
versiones_entorno = {}

for libreria in ["numpy", "pandas", "matplotlib", "sklearn"]:
    try:
        modulo = importlib.import_module(libreria)
        versiones_entorno[libreria] = modulo.__version__
    except ImportError:
        versiones_entorno[libreria] = None

ENTORNO = {
    "interprete": sys.executable,
    "entorno_virtual": (
        ".venv" in Path(sys.executable).parts
        or sys.prefix != sys.base_prefix
    ),
    "carpeta_trabajo": str(Path.cwd()),
    "versiones": versiones_entorno,
}

print("[OK] Información del entorno guardada en ENTORNO.")

## 3. Estructura del repositorio y artefactos

Esta etapa comprueba que el repositorio posee las carpetas y los archivos
definidos para el proyecto.

La estructura ya fue creada previamente, por lo que el código no genera ni
sobrescribe elementos. Solamente verifica su existencia y muestra su estado.
Los datos originales se conservan en `F1/data/raw/`, mientras que los resultados
derivados de la Fase 2 se almacenarán en `F1/data/processed/`.

In [ ]:
# Buscar la raíz del proyecto desde la carpeta actual o sus carpetas superiores.
actual = Path.cwd().resolve()

RAIZ = next(
    (
        carpeta
        for carpeta in [actual, *actual.parents]
        if (carpeta / "README.md").is_file()
        and (carpeta / "F1/data/raw/ai4i2020.csv").is_file()
    ),
    None,
)

if RAIZ is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto ni el archivo ai4i2020.csv."
    )

RUTAS_PROYECTO = {
    "datos originales": RAIZ / "F1/data/raw",
    "datos procesados": RAIZ / "F1/data/processed",
    "notebooks F1": RAIZ / "F1/notebooks",
    "documentación F1": RAIZ / "F1/docs",
    "código reutilizable F1": RAIZ / "F1/src",
    "fase 2": RAIZ / "F2",
    "fase 3": RAIZ / "F3",
    "fase 4": RAIZ / "F4",
}

ARCHIVOS_PRINCIPALES = {
    "README": RAIZ / "README.md",
    "dependencias": RAIZ / "requirements.txt",
    "reglas de Git": RAIZ / ".gitignore",
    "dataset original": RAIZ / "F1/data/raw/ai4i2020.csv",
    "notebook de definición": RAIZ / "F1/notebooks/F1_Definicion.ipynb",
    "notebook validador": RAIZ / "F1/notebooks/validador.ipynb",
}

print("Carpeta inicial :", actual)
print("Raíz encontrada :", RAIZ)

print("\nCarpetas")
for descripcion, ruta in RUTAS_PROYECTO.items():
    estado = "[OK]" if ruta.is_dir() else "[FALTA]"
    ruta_visible = ruta.relative_to(RAIZ)
    print(f"  {estado:8} {descripcion:28} {ruta_visible}")

print("\nArchivos principales")
for descripcion, ruta in ARCHIVOS_PRINCIPALES.items():
    estado = "[OK]" if ruta.is_file() else "[FALTA]"
    ruta_visible = ruta.relative_to(RAIZ)
    print(f"  {estado:8} {descripcion:28} {ruta_visible}")

In [ ]:
carpetas_faltantes = [
    ruta.relative_to(RAIZ).as_posix()
    for ruta in RUTAS_PROYECTO.values()
    if not ruta.is_dir()
]

archivos_faltantes = [
    ruta.relative_to(RAIZ).as_posix()
    for ruta in ARCHIVOS_PRINCIPALES.values()
    if not ruta.is_file()
]

assert not carpetas_faltantes, (
    f"Faltan carpetas del proyecto: {carpetas_faltantes}"
)

assert not archivos_faltantes, (
    f"Faltan archivos principales: {archivos_faltantes}"
)

print("[OK] La estructura principal del proyecto está completa.")

### 3.1 Archivo `.gitignore`

El archivo `.gitignore` indica qué elementos locales o regenerables no deben
incorporarse al historial de Git.

En este proyecto se excluyen el entorno virtual, las cachés de Python y Jupyter,
los archivos temporales del sistema operativo y los archivos que podrían contener
credenciales. El dataset AI4I 2020 se conserva porque su tamaño permite
versionarlo y facilita la reproducción del análisis.


In [ ]:
ARCHIVO_GITIGNORE = RAIZ / ".gitignore"

REGLAS_REQUERIDAS = {
    ".venv/",
    "__pycache__/",
    "*.pyc",
    ".ipynb_checkpoints/",
    "Thumbs.db",
    "desktop.ini",
    ".env",
}

if not ARCHIVO_GITIGNORE.is_file():
    raise FileNotFoundError("No se encontró .gitignore en la raíz.")

contenido_gitignore = ARCHIVO_GITIGNORE.read_text(
    encoding="utf-8"
)

reglas_activas = {
    linea.strip()
    for linea in contenido_gitignore.splitlines()
    if linea.strip() and not linea.strip().startswith("#")
}

reglas_faltantes = REGLAS_REQUERIDAS - reglas_activas

print("Archivo:", ARCHIVO_GITIGNORE.relative_to(RAIZ))
print("Reglas activas:", len(reglas_activas))

if reglas_faltantes:
    print("\nReglas pendientes:")
    for regla in sorted(reglas_faltantes):
        print("  ·", regla)
else:
    print("[OK] .gitignore contiene todas las reglas requeridas.")

### 3.2 Archivo `requirements.txt`

El archivo `requirements.txt` registra las dependencias necesarias para
reproducir el entorno del proyecto.

En esta sección se comprueba que el archivo existe, que las dependencias
principales tienen una versión declarada y que esas versiones corresponden
con las instaladas en el entorno virtual.

In [ ]:
from importlib.metadata import (
    version as version_instalada,
    PackageNotFoundError,
)

ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"

if not ARCHIVO_REQUIREMENTS.is_file():
    raise FileNotFoundError(
        "No se encontró requirements.txt en la raíz del proyecto."
    )

lineas_requirements = [
    linea.strip()
    for linea in ARCHIVO_REQUIREMENTS.read_text(
        encoding="utf-8"
    ).splitlines()
    if linea.strip() and not linea.strip().startswith("#")
]

dependencias_declaradas = {}

for linea in lineas_requirements:
    if "==" in linea:
        nombre, version = linea.split("==", maxsplit=1)
        dependencias_declaradas[nombre.lower()] = version

DEPENDENCIAS_PRINCIPALES = [
    "numpy",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "jupyterlab",
    "notebook",
    "ipykernel",
]

print("Archivo:", ARCHIVO_REQUIREMENTS.relative_to(RAIZ))
print("Dependencias declaradas:", len(lineas_requirements))
print("\nVerificación de dependencias principales")

faltantes_requirements = []

for paquete in DEPENDENCIAS_PRINCIPALES:
    declarada = dependencias_declaradas.get(paquete)

    if declarada is None:
        faltantes_requirements.append(paquete)
        print(f"  [FALTA] {paquete}")
        continue

    try:
        instalada = version_instalada(paquete)
        coincide = instalada == declarada
        estado = "[OK]" if coincide else "[REVISAR]"

        print(
            f"  {estado:9} {paquete:15} "
            f"declarada={declarada} instalada={instalada}"
        )

    except PackageNotFoundError:
        print(
            f"  [FALTA] {paquete:15} "
            f"declarada={declarada}, pero no está instalada"
        )

assert not faltantes_requirements, (
    "Faltan dependencias principales en requirements.txt: "
    f"{faltantes_requirements}"
)

## 4. Módulo reutilizable

Esta sección crea un módulo con funciones para localizar la raíz del proyecto y
comprobar que un conjunto de columnas contiene las variables esperadas.

El módulo podrá reutilizarse en los notebooks de las fases posteriores.

In [ ]:
DIR_SRC = RAIZ / "F1/src"
ARCHIVO_UTILIDADES = DIR_SRC / "utilidades.py"

CODIGO_UTILIDADES = '''\
"""Funciones reutilizables del proyecto MCDI500 Grupo 9."""

from pathlib import Path


class ProyectoError(Exception):
    """Error específico de validación del proyecto."""


def buscar_raiz(inicio="."):
    """Busca la raíz que contiene README.md y la carpeta F1."""
    actual = Path(inicio).resolve()

    for carpeta in [actual, *actual.parents]:
        if (carpeta / "README.md").is_file() and (carpeta / "F1").is_dir():
            return carpeta

    raise ProyectoError("No se encontró la raíz del proyecto.")


def validar_columnas(columnas, esperadas):
    """Comprueba que estén presentes todas las columnas esperadas."""
    faltantes = sorted(set(esperadas) - set(columnas))

    if faltantes:
        raise ProyectoError(f"Faltan columnas: {faltantes}")

    return True
'''

if ARCHIVO_UTILIDADES.exists():
    print("[AVISO] El archivo ya existe y no será sobrescrito.")
else:
    ARCHIVO_UTILIDADES.write_text(
        CODIGO_UTILIDADES,
        encoding="utf-8",
    )
    print(
        "[OK] Módulo creado:",
        ARCHIVO_UTILIDADES.relative_to(RAIZ),
    )

In [ ]:
if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))

import utilidades

COLUMNAS_ESPERADAS = [
    "UDI",
    "Product ID",
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

assert utilidades.buscar_raiz(Path.cwd()) == RAIZ
assert utilidades.validar_columnas(
    COLUMNAS_ESPERADAS,
    COLUMNAS_ESPERADAS,
)

print("[OK] F1/src/utilidades.py es importable.")
print("[OK] Las pruebas normales del módulo fueron superadas.")

### 4.1 Pruebas del módulo

El módulo se verifica mediante un caso normal, un caso límite y una excepción
controlada.

Estas pruebas permiten comprobar que las funciones responden correctamente
cuando reciben las columnas esperadas, cuando reciben una entrada vacía válida
y cuando falta una variable obligatoria.

In [ ]:
# Caso normal: están presentes todas las columnas requeridas.
assert utilidades.validar_columnas(
    COLUMNAS_ESPERADAS,
    COLUMNAS_ESPERADAS,
) is True

assert utilidades.buscar_raiz(Path.cwd()) == RAIZ

print("[OK] Caso normal: raíz y columnas verificadas.")


# Caso límite: no se exige ninguna columna y la entrada está vacía.
assert utilidades.validar_columnas([], []) is True

print("[OK] Caso límite: listas vacías aceptadas.")


# Excepción: falta una columna obligatoria.
columnas_incompletas = [
    columna
    for columna in COLUMNAS_ESPERADAS
    if columna != "Torque [Nm]"
]

try:
    utilidades.validar_columnas(
        columnas_incompletas,
        COLUMNAS_ESPERADAS,
    )

    raise AssertionError(
        "La prueba debía detectar la ausencia de Torque [Nm]."
    )

except utilidades.ProyectoError as error:
    print("[OK] Excepción controlada:", error)

## 5. Selección y ficha del conjunto de datos

Esta sección documenta la procedencia de AI4I 2020 y sus características
principales. La ficha permite identificar la fuente, la licencia, el tamaño y
las condiciones bajo las cuales puede utilizarse.

### 5.1 Procedencia

| Campo | Valor |
|---|---|
| Título | AI4I 2020 Predictive Maintenance Dataset |
| Repositorio | UCI Machine Learning Repository |
| Año | 2020 |
| Naturaleza | Conjunto de datos sintético |
| Registros | 10.000 |
| Columnas del archivo | 14 |
| Valores faltantes reportados | No |
| Licencia | Creative Commons Attribution 4.0 International |
| DOI | https://doi.org/10.24432/C5HS5C |
| Archivo local | `F1/data/raw/ai4i2020.csv` |

El conjunto fue diseñado para representar condiciones de mantenimiento
predictivo industrial. Incluye seis variables predictoras, identificadores,
una variable de falla general y cinco indicadores de modos de falla.

**Referencia APA 7**

*AI4I 2020 Predictive Maintenance Dataset* [Conjunto de datos]. (2020).
UCI Machine Learning Repository. https://doi.org/10.24432/C5HS5C

In [ ]:
FICHA = {
    "titulo": "AI4I 2020 Predictive Maintenance Dataset",
    "repositorio": "UCI Machine Learning Repository",
    "anio": 2020,
    "url": (
        "https://archive.ics.uci.edu/dataset/601/"
        "ai4i+2020+predictive+maintenance+dataset"
    ),
    "doi": "https://doi.org/10.24432/C5HS5C",
    "licencia": "CC BY 4.0",
    "archivo": RAIZ / "F1/data/raw/ai4i2020.csv",
    "filas": 10000,
    "columnas": 14,
    "sintetico": True,
    "faltantes_reportados": False,
}

print("Conjunto :", FICHA["titulo"])
print("Fuente   :", FICHA["repositorio"])
print("Archivo  :", FICHA["archivo"].relative_to(RAIZ))
print("Tamaño   :", FICHA["filas"], "filas ×", FICHA["columnas"], "columnas")
print("Licencia :", FICHA["licencia"])

### 5.2 Evaluación contra los criterios del curso

El conjunto se evalúa antes de utilizarlo para comprobar que posee suficientes
registros, columnas y tipos de variables. También se revisan valores faltantes,
duplicados y tamaño del archivo.

Esta comprobación complementa el trabajo realizado en `validador.ipynb`.

In [ ]:
ROLES_VARIABLES = {
    "UDI": "identificador",
    "Product ID": "identificador",
    "Type": "nominal",
    "Air temperature [K]": "continua",
    "Process temperature [K]": "continua",
    "Rotational speed [rpm]": "continua",
    "Torque [Nm]": "continua",
    "Tool wear [min]": "discreta",
    "Machine failure": "binaria",
    "TWF": "binaria",
    "HDF": "binaria",
    "PWF": "binaria",
    "OSF": "binaria",
    "RNF": "binaria",
}


def evaluar_criterios(df, ruta, roles):
    """Evalúa el dataset contra los requisitos mínimos del curso."""

    filas, columnas = df.shape
    max_nulos = df.isna().mean().max() * 100
    duplicados = int(df.duplicated().sum())
    peso_mb = ruta.stat().st_size / 1024**2

    roles_utiles = {
        rol
        for rol in roles.values()
        if rol != "identificador"
    }

    numericas = sum(
        rol in {"continua", "discreta"}
        for rol in roles.values()
    )

    categoricas = sum(
        rol in {"nominal", "binaria", "ordinal"}
        for rol in roles.values()
    )

    comprobaciones = [
        ("Al menos 2000 filas", filas >= 2000, f"{filas} filas"),
        ("Al menos 12 columnas", columnas >= 12, f"{columnas} columnas"),
        (
            "Al menos 3 roles analíticos",
            len(roles_utiles) >= 3,
            ", ".join(sorted(roles_utiles)),
        ),
        (
            "Al menos una variable numérica",
            numericas >= 1,
            f"{numericas} variables",
        ),
        (
            "Al menos una variable categórica",
            categoricas >= 1,
            f"{categoricas} variables",
        ),
        (
            "Ninguna variable supera 60% de nulos",
            max_nulos <= 60,
            f"máximo {max_nulos:.1f}%",
        ),
        (
            "Archivo menor de 100 MB",
            peso_mb <= 100,
            f"{peso_mb:.1f} MB",
        ),
        (
            "Sin filas duplicadas exactas",
            duplicados == 0,
            f"{duplicados} duplicadas",
        ),
    ]

    resultado = pd.DataFrame(
        [
            {
                "estado": "OK" if cumple else "FALTA",
                "requisito": requisito,
                "detalle": detalle,
            }
            for requisito, cumple, detalle in comprobaciones
        ]
    )

    return resultado

In [ ]:
df_candidato = pd.read_csv(
    FICHA["archivo"],
    encoding="utf-8-sig",
)

evaluacion = evaluar_criterios(
    df_candidato,
    FICHA["archivo"],
    ROLES_VARIABLES,
)

display(evaluacion)

assert "FALTA" not in evaluacion["estado"].values, (
    "El dataset no cumple todos los requisitos mínimos."
)

print("[OK] AI4I 2020 cumple los requisitos mínimos del curso.")

### 5.3 Diccionario de variables

El diccionario identifica el significado, el rol analítico, la unidad y el uso
previsto de cada variable.

Esta clasificación evita tratar los identificadores como mediciones y permite
excluir los modos de falla como predictores de `Machine failure`, evitando fuga
de información.

In [ ]:
DICCIONARIO = [
    ("UDI", "identificador", "Identificador único del registro", None, "No"),
    ("Product ID", "identificador", "Identificador del producto", None, "No"),
    ("Type", "nominal", "Tipo de producto: L, M o H", None, "Sí"),
    (
        "Air temperature [K]",
        "continua",
        "Temperatura del aire",
        "K",
        "Sí",
    ),
    (
        "Process temperature [K]",
        "continua",
        "Temperatura del proceso",
        "K",
        "Sí",
    ),
    (
        "Rotational speed [rpm]",
        "continua",
        "Velocidad de rotación",
        "rpm",
        "Sí",
    ),
    (
        "Torque [Nm]",
        "continua",
        "Torque aplicado",
        "Nm",
        "Sí",
    ),
    (
        "Tool wear [min]",
        "discreta",
        "Desgaste acumulado de la herramienta",
        "min",
        "Sí",
    ),
    (
        "Machine failure",
        "binaria",
        "Variable objetivo: 1 indica falla",
        None,
        "Objetivo",
    ),
    (
        "TWF",
        "binaria",
        "Falla por desgaste de herramienta",
        None,
        "No",
    ),
    (
        "HDF",
        "binaria",
        "Falla por disipación de calor",
        None,
        "No",
    ),
    (
        "PWF",
        "binaria",
        "Falla de potencia",
        None,
        "No",
    ),
    (
        "OSF",
        "binaria",
        "Falla por sobreesfuerzo",
        None,
        "No",
    ),
    (
        "RNF",
        "binaria",
        "Falla aleatoria",
        None,
        "No",
    ),
]

diccionario = pd.DataFrame(
    DICCIONARIO,
    columns=[
        "variable",
        "rol_analitico",
        "descripcion",
        "unidad",
        "uso_como_predictor",
    ],
)

assert set(diccionario["variable"]) == set(df_candidato.columns), (
    "El diccionario no coincide con las columnas del dataset."
)

assert len(diccionario) == FICHA["columnas"], (
    "El diccionario no contiene las 14 variables esperadas."
)

print(
    "[OK] Diccionario completo:",
    len(diccionario),
    "variables.",
)

display(diccionario)

### 5.4 Comprobación del archivo original

Esta sección comprueba que el archivo original está disponible y que sus
dimensiones y columnas coinciden con la ficha y el diccionario.

La revisión no modifica, limpia ni transforma los datos. Solo verifica la
entrada que utilizará la Fase 2.

In [ ]:
ARCHIVO = FICHA["archivo"]

if not ARCHIVO.is_file():
    raise FileNotFoundError(
        f"No se encontró el dataset en: {ARCHIVO}"
    )

df_crudo = pd.read_csv(
    ARCHIVO,
    encoding="utf-8-sig",
)

print("Archivo encontrado :", ARCHIVO.relative_to(RAIZ))
print("Dimensiones reales :", df_crudo.shape)
print(
    "Dimensiones esperadas:",
    (FICHA["filas"], FICHA["columnas"]),
)

assert df_crudo.shape == (
    FICHA["filas"],
    FICHA["columnas"],
), "Las dimensiones no coinciden con la ficha."

utilidades.validar_columnas(
    df_crudo.columns,
    diccionario["variable"],
)

print("[OK] Dimensiones y columnas verificadas.")
print("Valores faltantes :", int(df_crudo.isna().sum().sum()))
print("Filas duplicadas  :", int(df_crudo.duplicated().sum()))

print("\nDistribución de Machine failure")
display(
    df_crudo["Machine failure"]
    .value_counts()
    .rename_axis("clase")
    .to_frame("registros")
)

## 6. Control de versiones

Esta sección consulta el estado del repositorio Git para dejar evidencia de la
rama activa, la identidad configurada, los últimos commits y los cambios locales.


El notebook solo consulta información. No ejecuta `git add`, `git commit`,
`git push` ni modifica el historial.

In [ ]:
def consultar_git(*argumentos):
    """Ejecuta una consulta de Git dentro del repositorio."""

    if shutil.which("git") is None:
        return False, "Git no está disponible en el sistema."

    try:
        proceso = subprocess.run(
            ["git", "-C", str(RAIZ), *argumentos],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=20,
        )

    except (OSError, subprocess.SubprocessError) as error:
        return False, f"No se pudo consultar Git: {error}"

    if proceso.returncode != 0:
        mensaje = proceso.stderr.strip() or "Git terminó con error."
        return False, mensaje

    return True, proceso.stdout.strip()
    

In [ ]:
ok, version_git = consultar_git("--version")
print("Git:", version_git if ok else f"[AVISO] {version_git}")

ok, usuario = consultar_git("config", "user.name")
print("Usuario:", usuario if ok and usuario else "[FALTA]")

ok, correo = consultar_git("config", "user.email")
print("Correo:", correo if ok and correo else "[FALTA]")

ok, rama = consultar_git("branch", "--show-current")
print("Rama activa:", rama if ok and rama else "[FALTA]")

ok, remoto = consultar_git("remote", "get-url", "origin")
print("Repositorio remoto:", remoto if ok else f"[AVISO] {remoto}")

ok, historial = consultar_git("log", "--oneline", "-5")
print("\nÚltimos commits")
print(historial if ok and historial else "[AVISO] Sin commits disponibles")

ok, estado = consultar_git("status", "--short")
print("\nCambios locales")
print(estado if ok and estado else "Ningún cambio pendiente")


## 7. Validación técnica de la Fase 1

Esta sección reúne las comprobaciones principales de la fase. Si alguna
condición no se cumple, `assert` detiene la ejecución e indica qué elemento
debe corregirse.

Se verifica la estructura del repositorio, los archivos reproducibles, la
definición del proyecto, el dataset, el diccionario y el módulo reutilizable.

In [ ]:
def validar_fase1(
    raiz,
    proyecto,
    ficha,
    dataframe,
    diccionario_variables,
    evaluacion_dataset,
):
    """Comprueba los entregables técnicos principales de la Fase 1."""

    print("VALIDACIÓN DE LA FASE 1")
    print("-" * 50)

    # 1. Estructura
    carpetas = [
        "F1/data/raw",
        "F1/data/processed",
        "F1/notebooks",
        "F1/docs",
        "F1/src",
        "F2",
        "F3",
        "F4",
    ]

    faltan_carpetas = [
        carpeta
        for carpeta in carpetas
        if not (raiz / carpeta).is_dir()
    ]

    assert not faltan_carpetas, (
        f"Faltan carpetas: {faltan_carpetas}"
    )

    print(f"[OK] Estructura: {len(carpetas)} carpetas verificadas.")

    # 2. Archivos reproducibles
    archivos = [
        "README.md",
        "requirements.txt",
        ".gitignore",
        "F1/data/raw/ai4i2020.csv",
        "F1/notebooks/F1_Definicion.ipynb",
        "F1/src/utilidades.py",
    ]

    faltan_archivos = [
        archivo
        for archivo in archivos
        if not (raiz / archivo).is_file()
    ]

    assert not faltan_archivos, (
        f"Faltan archivos: {faltan_archivos}"
    )

    print(f"[OK] Archivos: {len(archivos)} elementos verificados.")

    # 3. Definición del proyecto
    claves = [
        "titulo",
        "problematica",
        "objetivo_general",
        "objetivos_especificos",
        "preguntas",
        "alcance",
        "limitaciones",
    ]

    faltan_claves = [
        clave
        for clave in claves
        if not proyecto.get(clave)
    ]

    assert not faltan_claves, (
        f"Faltan elementos en PROYECTO: {faltan_claves}"
    )

    print("[OK] Definición del proyecto completa.")

    # 4. Dataset
    assert dataframe.shape == (
        ficha["filas"],
        ficha["columnas"],
    ), "Las dimensiones del dataset no coinciden con la ficha."

    print(
        "[OK] Dataset:",
        dataframe.shape[0],
        "filas ×",
        dataframe.shape[1],
        "columnas.",
    )

    # 5. Diccionario
    assert set(diccionario_variables["variable"]) == set(
        dataframe.columns
    ), "El diccionario no coincide con el dataset."

    print(
        "[OK] Diccionario:",
        len(diccionario_variables),
        "variables documentadas.",
    )

    # 6. Evaluación
    assert "FALTA" not in evaluacion_dataset["estado"].values, (
        "El dataset no cumple todos los criterios mínimos."
    )

    print("[OK] Evaluación del dataset aprobada.")

    # 7. Módulo
    assert callable(utilidades.buscar_raiz)
    assert callable(utilidades.validar_columnas)

    print("[OK] Módulo utilidades importable.")

    print("\n[OK] Validación principal de la Fase 1 completada.")

    return True

In [ ]:
FASE1_VALIDA = validar_fase1(
    RAIZ,
    PROYECTO,
    FICHA,
    df_crudo,
    diccionario,
    evaluacion,
)

## 8. Vinculación con el mapa conceptual

El mapa conceptual de la Formativa 1 representa el flujo desde la definición
del problema hasta la obtención de resultados reproducibles.

La tabla siguiente relaciona sus elementos con la implementación real del
proyecto y distingue lo que ya fue desarrollado de lo que queda proyectado
para fases posteriores.

In [ ]:
VINCULACION = [
    (
        "Delimitar el problema",
        "Implementado",
        "F1_Definicion.ipynb y README.md",
    ),
    (
        "Validar y describir los datos",
        "Implementado",
        "validador.ipynb e informe_dataset.md",
    ),
    (
        "Preservar los datos originales",
        "Implementado",
        "F1/data/raw/ai4i2020.csv",
    ),
    (
        "Guardar datos procesados",
        "Proyectado para F2",
        "F1/data/processed/",
    ),
    (
        "Preparar el entorno reproducible",
        "Implementado",
        ".venv, requirements.txt y kernel del proyecto",
    ),
    (
        "Analizar en Jupyter",
        "Parcial",
        "Reconocimiento inicial en F1; preparación y exploración en F2",
    ),
    (
        "Documentar la evidencia",
        "Implementado",
        "README.md, notebooks y F1/docs/",
    ),
    (
        "Registrar cambios con Git",
        "Implementado",
        "Historial de commits del repositorio",
    ),
    (
        "Colaborar mediante GitHub",
        "Adaptado a trabajo individual",
        "Repositorio remoto personal",
    ),
    (
        "Modelar y evaluar",
        "Proyectado",
        "Fases 3 y 4",
    ),
]

vinculacion = pd.DataFrame(
    VINCULACION,
    columns=[
        "elemento_del_mapa",
        "estado",
        "evidencia",
    ],
)

assert vinculacion["evidencia"].str.strip().ne("").all(), (
    "Existen elementos del mapa sin evidencia."
)

print(
    "[OK]",
    len(vinculacion),
    "elementos del mapa vinculados.",
)

display(vinculacion)

## 9. Persistencia y trazabilidad

Los resultados de las comprobaciones deben quedar disponibles fuera del
notebook. Por esta razón, el diccionario de variables, la evaluación del
dataset, la vinculación con el mapa y los metadatos de ejecución se guardan
en `F1/docs/`.

Estos archivos permiten revisar las decisiones y las condiciones de ejecución
sin tener que localizar cada resultado dentro del notebook.

In [ ]:
DIR_DOCS = RAIZ / "F1/docs"

assert DIR_DOCS.is_dir(), (
    "No se encontró la carpeta F1/docs."
)

ARCHIVO_DICCIONARIO = (
    DIR_DOCS / "diccionario_variables.csv"
)

ARCHIVO_EVALUACION = (
    DIR_DOCS / "evaluacion_criterios_dataset.csv"
)

ARCHIVO_VINCULACION = (
    DIR_DOCS / "vinculacion_mapa_conceptual.csv"
)

ARCHIVO_METADATOS = (
    DIR_DOCS / "metadatos_fase1.json"
)

diccionario.to_csv(
    ARCHIVO_DICCIONARIO,
    index=False,
    encoding="utf-8-sig",
)

evaluacion.to_csv(
    ARCHIVO_EVALUACION,
    index=False,
    encoding="utf-8-sig",
)

vinculacion.to_csv(
    ARCHIVO_VINCULACION,
    index=False,
    encoding="utf-8-sig",
)

METADATOS = {
    "proyecto": PROYECTO["titulo"],
    "grupo": PROYECTO["grupo"],
    "integrantes": PROYECTO["integrantes"],
    "fase": "F1",
    "fecha_ejecucion": date.today().isoformat(),
    "semilla": SEMILLA,
    "python": sys.version.split()[0],
    "sistema": f"{platform.system()} {platform.release()}",
    "interprete": ENTORNO["interprete"],
    "entorno_virtual": ENTORNO["entorno_virtual"],
    "dependencias": ENTORNO["versiones"],
    "dataset": {
        "titulo": FICHA["titulo"],
        "archivo": FICHA["archivo"].relative_to(RAIZ).as_posix(),
        "filas": int(df_crudo.shape[0]),
        "columnas": int(df_crudo.shape[1]),
        "licencia": FICHA["licencia"],
        "doi": FICHA["doi"],
    },
}

ARCHIVO_METADATOS.write_text(
    json.dumps(
        METADATOS,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Archivos generados:")

for archivo in [
    ARCHIVO_DICCIONARIO,
    ARCHIVO_EVALUACION,
    ARCHIVO_VINCULACION,
    ARCHIVO_METADATOS,
]:
    print(
        "  [OK]",
        archivo.relative_to(RAIZ),
        f"({archivo.stat().st_size} bytes)",
    )

## Conclusiones y trazabilidad

La Fase 1 permitió definir la problemática de mantenimiento predictivo, formular
la pregunta de investigación y establecer los objetivos del proyecto. También
se comprobó el entorno de ejecución, la estructura del repositorio y la
disponibilidad de las dependencias necesarias.

El archivo `ai4i2020.csv` contiene 10.000 registros y 14 columnas. La validación
inicial no detectó valores faltantes ni filas duplicadas exactas. Las variables
fueron clasificadas según su rol analítico y documentadas en un diccionario.

Los identificadores `UDI` y `Product ID` no serán utilizados como predictores.
Los indicadores `TWF`, `HDF`, `PWF`, `OSF` y `RNF` tampoco se utilizarán para
predecir `Machine failure`, porque podrían producir fuga de información.

La trazabilidad se mantiene mediante el notebook, el README, el historial de
Git y los archivos generados en `F1/docs/`. La limpieza y transformación de los
datos quedan reservadas para la Fase 2.

## 10. Reflexión técnica

### Hallazgos

- AI4I 2020 cumple los requisitos mínimos de tamaño y variedad de variables.
- El conjunto no presenta valores faltantes ni filas duplicadas exactas.
- El tipo almacenado por Python no determina por sí solo el rol analítico de
  una variable. Por ejemplo, `UDI` es numérica, pero funciona como identificador.
- La variable `Machine failure` está desbalanceada, por lo que será necesario
  considerar esta condición durante la evaluación posterior.
- El carácter sintético del conjunto limita la generalización de los resultados
  a maquinaria industrial real.

### Dificultades y soluciones

- Las rutas relativas fallaron inicialmente porque Jupyter se estaba ejecutando
  desde `F1/notebooks`. El problema se resolvió mediante una función que busca
  la raíz del proyecto.
- El entorno se verificó comprobando que el intérprete se encuentra dentro de
  `.venv` y registrando las versiones de las librerías.
- La información del proyecto se organizó en estructuras reutilizables para
  mantener coherencia entre validaciones y archivos de documentación.

### Decisiones pendientes

- Definir en la Fase 2 qué transformaciones son realmente necesarias, evitando
  aplicar técnicas sin una justificación basada en los datos.
- Determinar la estrategia de codificación para la variable `Type`.
- Decidir si las variables numéricas requieren escalamiento según el algoritmo
  utilizado en fases posteriores.
- Seleccionar métricas apropiadas para evaluar la variable objetivo
  desbalanceada.

## 11. Verificación antes de entregar

### Cuaderno

- [x] Corre completo tras `Kernel → Restart Kernel and Run All Cells`.
- [x] La numeración de ejecución es continua.
- [x] Cada bloque de código tiene una explicación narrativa.
- [x] Las rutas se construyen desde la raíz del proyecto.
- [x] La semilla está declarada.

### Contenido técnico

- [x] La problemática y los objetivos están declarados.
- [x] El alcance y las limitaciones están declarados.
- [x] El entorno se verifica con evidencia.
- [x] La procedencia de AI4I 2020 está documentada.
- [x] El diccionario declara el rol de las 14 variables.
- [x] El conjunto de datos fue evaluado según los criterios del curso.
- [x] El módulo incluye pruebas normales, límite y de excepción.

### Archivos

- [x] `.gitignore` y `requirements.txt` tienen contenido.
- [x] `README.md` documenta el proyecto.
- [x] `F1/docs/` contiene el diccionario, la evaluación, la vinculación y los metadatos.
- [x] `F1/src/utilidades.py` es importable.

### Repositorio

- [x] La carpeta `F1` contiene el notebook y sus documentos.
- [x] Git está configurado con el nombre y correo del autor.
- [x] El notebook ejecutado y los archivos generados están registrados en un commit.
- [x] Los commits locales están enviados a GitHub.